In [1]:
import pandas as pd
import os
import probabilistic_automaton as pa # to create the probabilistic automaton for sampling
import vectorization as vec # to create entropy vectors for data encoding/preprocessing
import sampling 

import itertools
from pyranges.readers import read_gtf
from processing_fasta import *
from collections import defaultdict

from Bio import SeqIO

In [2]:

gtf_file_path = '/data/gencode.v49lift37.basic.annotation_protein_coding.gtf' # to build on the automata
cwd = Path(os.getcwd())
print("Loading GTF database into memory, please wait...")
gtf = read_gtf(str(cwd) + gtf_file_path, as_df=True)
print("GTF loaded successfully!")

Loading GTF database into memory, please wait...
GTF loaded successfully!


In [3]:

#Getting the number of genes the 22 chromosomes and the X and Y chromosomes
chr_list = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9',
            'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrX', 'chrY']
for chr in chr_list:
    gtf_filtered = gtf[gtf['Chromosome'] == chr]
    number_genes = len(gtf_filtered)
    print(f'number of genes on {chr}: ', number_genes)
    

number of genes on chr1:  2098
number of genes on chr2:  1257
number of genes on chr3:  1085
number of genes on chr4:  762
number of genes on chr5:  897
number of genes on chr6:  1049
number of genes on chr7:  937
number of genes on chr8:  710
number of genes on chr9:  797
number of genes on chr10:  743
number of genes on chr11:  1318
number of genes on chr12:  1042
number of genes on chr13:  321
number of genes on chr14:  619
number of genes on chr15:  607
number of genes on chr16:  860
number of genes on chr17:  1190
number of genes on chr18:  267
number of genes on chr19:  1483
number of genes on chr20:  549
number of genes on chr21:  230
number of genes on chr22:  449
number of genes on chrX:  876
number of genes on chrY:  63


## General Sampling: first try ( code too heavy not to run)
This code takes in: list of chromosome names 
what it does : Creates a folder named "generated_sequences" 
               Creates a folder for each Chromosome in "generated_sequences" 
               For each Chromosome : gets the number of its genes 
                                    For each gene: creates an automata for that gene 
                                                   generates N samples for that gene 
                                                   saves each sample in fasta file in the chromosome's folder
Return: CSV file with the following structure:sample_id, chromosome, gene_location, sequence_length, acceptance_status, acceptance_score, file_path

Test1: Running the code only on Chromosome 22
       The chromosome has: 449 genes
       Trying to Generate N = 100 samples for each gene
       ((( 449 automata and 44 900 generation )))
Results: more than 30 minutes of running 

Test2: Running the code only on Chromosome 22
       The chromosome has: 449 genes BUT WE SELECT THE FIRST 10 GENES 
       Trying to Generate N = 100 samples for each gene
Results: 3 minutes of running. Samples well Generated

Solution? Try MultiProcessing not sure there are no matrix

In [ ]:
database_rows = [] #CSV file 
chr_list = ['chr22'] # for the moment running only on chr22, but we can run it on all chromosomes later

for chr in chr_list:
    os.makedirs(f"generated_sequences/{chr}", exist_ok=True) #folder for each chromosome
    
    #Getting the number of genes on this chromosome
    gtf_filtered = gtf[gtf['Chromosome'] == chr]
    number_genes = len(gtf_filtered)
    
    #Generating 100 samples for each gene on the chromosome
    for location in range(1, number_genes + 1): 
        my_automata = pa.automata_builder(gtf_file_path, chr, gene_number = location) # creating the automata
        N = 5 # number of samples to generate per gene
        for i in range(N):
            sample = sampling.mutated_sample(id = i, chromosome = chr, location = location, sequence = [], automata = my_automata)
            sample.generate_mutated_sample()
            acceptance_status, score = my_automata.accepts(sample.sequence)

            # create a text file for the generated sequence
            file_name = f"{location}_sample_{i}.fasta"
            file_path = f"generated_sequences/{chr}/{file_name}"
            # save the generated sequence in a fasta file
            with open(file_path, "w") as f:
                f.write(f">{location}_sample_{i}\n") # Standard FASTA header
                f.write(sample.sequence)
            
            # save the metadata AND the file path to our database list
            database_rows.append({
                "sample_id": i,
                "chromosome": chr,
                "gene_location": location, 
                "sequence_length": len(sample.sequence),
                "acceptance_status": acceptance_status,
                "acceptance_score": score,
                "file_path": f"{chr}/{file_name}"
            })

            # export the clean, readable database to a CSV so we can use it after 
            df = pd.DataFrame(database_rows)
            df.to_csv("mutated_samples_database.csv", index=False)
            
print(f"The dataset for th {chr_list} is ready!")



## Focused Sampling : Chromosome 22 gene number 10 ==> 5 000 samples 
## Approach 1:  Without controling the length of the samples 
Why focused sampling ? We are choosing one single gene from one Chromosome 
                       Generating N = 5 000 sample for that gene 

Reason : To run the CNN we need more than N = 100 sample for one single gene.
         

In [4]:

database_rows = [] #CSV file 
chr_list = ['chr22'] # for the moment running only on chr22, but we can run it on all chromosomes later
N = 5000 # number of samples to generate per gene

for chr in chr_list:
    os.makedirs(f"focused_sampling/{chr}", exist_ok=True) #folder for each chromosome
    
    #Getting the number of genes on this chromosome
    gtf_filtered = gtf[gtf['Chromosome'] == chr]
    number_genes = len(gtf_filtered)
    
    # Choosing a specific gene to focus on, for example gene number 10
    focused_gene_number = 10
    location = focused_gene_number
 
    # the one and only automata
    my_automata = pa.automata_builder(gtf_file_path, chr, gene_number = location) # creating the automata
    
    for i in range(N):
        sample = sampling.mutated_sample(id = i, chromosome = chr, location = location, sequence = [], automata = my_automata,status = True)
        # No length Control
        sample.generate_mutated_sample()
        acceptance_status, score = my_automata.accepts(sample.sequence)

        # create a text file for the generated sequence
        file_name = f"{location}_sample_{i}.fasta"
        file_path = f"focused_sampling/{chr}/{file_name}"
        # save the generated sequence in a fasta file
        with open(file_path, "w") as f:
                f.write(f">{location}_sample_{i}\n") # Standard FASTA header
                f.write(sample.sequence)
                
        # save the metadata AND the file path to our database list
        database_rows.append({
                        "sample_id": i,
                        "chromosome": chr,
                        "gene_location": location, 
                        "sequence_length": len(sample.sequence),
                        "sample_status": sample.status, # if using controlled length used the status can be false 
                        "acceptance_score": score,
                        "file_path": f"{chr}/{file_name}"
                })

    # export the clean, readable database to a CSV so we can use it after 
    df = pd.DataFrame(database_rows)
    ## named manually 
    df.to_csv("focused_sampling_chr22_g10.csv", index=False)
            
print(f"The dataset for the chromosome(s) {chr_list} gene number {location} is ready! with {N} samples")


KeyboardInterrupt: 

In [5]:
test3 = pd.read_csv("mutated_samples_database.csv")  
print(test3.head())

   sample_id chromosome  gene_location  sequence_length  acceptance_status  \
0          0      chr22             10             5999               True   
1          1      chr22             10            16272               True   
2          2      chr22             10            10718               True   
3          3      chr22             10            21867               True   
4          4      chr22             10              499               True   

   acceptance_score                file_path  
0       -906.322373  chr22/10_sample_0.fasta  
1      -2302.315596  chr22/10_sample_1.fasta  
2      -1552.326170  chr22/10_sample_2.fasta  
3      -3197.410436  chr22/10_sample_3.fasta  
4        -70.761060  chr22/10_sample_4.fasta  


In [6]:
# Check different lengths of the generated samples
test3["sequence_length"].value_counts()

sequence_length
34       67
947       2
15340     2
8720      2
2269      2
         ..
3282      1
19145     1
37465     1
26886     1
4979      1
Name: count, Length: 924, dtype: int64

In [12]:
# Filter on the length problem Check
counts = test3['sequence_length'].value_counts()

rep_len = counts[counts >= 2].index #extracting the lengths that are repititive (from all lengths)

filtered_df = test3[test3['sequence_length'].isin(rep_len)] 
print("the list of repetitive lengths is:")
print (rep_len)

the list of repetitive lengths is:
Index([34, 947, 15340, 8720, 2269, 8093, 16864, 50939, 18880, 30194, 1875], dtype='int64', name='sequence_length')


## Samples Repitition Diagnostic
- Samples of the same length can be doublons
- Run the check all the samples of the same length  
- Eliminate the doublons of samples of the same length
- In order to read fasta files we use the Biopython Library with implemented fonctionalities SeqIo.
#### Method: 
we create a dictionary with this format { "sequences" : ["file1.fasta", "file2.fasta"] }
meaning that the index dequence is represnted by multiple files 
#### Application:
on each repetitive length that we detect in the filtered_reptitive_length dataset.

In [ ]:
df1 = test3 # copy of the dataframe 
print("\n-----------------------------------------------------  Samples Repitition Diagnostic ----------------------------------------------------------")
for length in rep_len:
    filtered = test3.query(f"sequence_length == {length}") # dataframe with only sequences with the same length

    # dictionary regrouping files that present the same sequence
    files_by_sequence = {} 

    for file_name in filtered["file_path"]:
        
        # we join the folder name with the file_name : "./focused_sampling/nom_fichier.fasta"
        complete_path = os.path.join("./focused_sampling", file_name)
        
        try:
            
            # putting the sequence in a tuple and assigning it as a key to the dictionary 
            file_seq_i = tuple(str(record.seq).upper() for record in SeqIO.parse(complete_path, "fasta"))
            
            if file_seq_i in files_by_sequence:
                # if we alr saw this sequence we add its file_name to the list
                files_by_sequence[file_seq_i].append(file_name)
            else:
                # its the first time that we see this file name
                files_by_sequence[file_seq_i]= [file_name]
                
        except FileNotFoundError:
            print(f"file not found : {complete_path}")
            

        
#---------------------------Results---------------------------------------------------------------------------------------------------------------------------

    print(f" The length of {length} we have {len(files_by_sequence)} unique sequences vs {counts[length]} generated sequences  ")
    uniques_only = True

    for sequence, list in files_by_sequence.items():
        if len(list) > 1:
            tous_uniques = False
            print(f" These {len(list)} files are exactly the same")
            print(f"   -> {', '.join(list)}")
#-------------------------- Elimination of Detected Doublons from the datframe -----------------------------------------------------------------------------
            for i in range(len(list)):
                 complete_path = os.path.join("./focused_sampling", file_name)
                if os.path.exists(list[i]):
                     os.remove(list[i])
                     
                df = df[df["file_path"] != list[i]] # we remove the row from the df 
            

    if tous_uniques:
        print("The files are all unique")  
# having new inexed dataframe
df1 = df1.reset_index(drop=True)  


-----------------------------------------------------  Samples Repitition Diagnostic ----------------------------------------------------------
 The length of 34 we have 64 unique sequences vs 67 generated sequences  
 These 4 files are exactly the same
   -> chr22/10_sample_134.fasta, chr22/10_sample_451.fasta, chr22/10_sample_582.fasta, chr22/10_sample_832.fasta
 The length of 947 we have 2 unique sequences vs 2 generated sequences  
 The length of 15340 we have 2 unique sequences vs 2 generated sequences  
 The length of 8720 we have 2 unique sequences vs 2 generated sequences  
 The length of 2269 we have 2 unique sequences vs 2 generated sequences  
 The length of 8093 we have 1 unique sequences vs 2 generated sequences  
 These 2 files are exactly the same
   -> chr22/10_sample_147.fasta, chr22/10_sample_728.fasta
 The length of 16864 we have 2 unique sequences vs 2 generated sequences  
 The length of 50939 we have 2 unique sequences vs 2 generated sequences  
 The length of 18

##  Approach 2: Length Controlled Sampling 5 000 samples 1 000 long. 
###  All the samples are 1 000 Nucleotides long
Methodology :
samples that failed to attend 1000 on a final state have status == false 
to be removed from the dataset 
- results all the samples with 1k of length are status true 
- the other are status false 

We have a Pandas Dataframe holding : ..............
And a list of generated instances of the mutated_sample class ( to use for the next step)

In [5]:

database_fields = [] #CSV file 
chr_list = ['chr22'] # for the moment running only on chr22, but we can run it on all chromosomes later
N = 5000 # number of samples to generate per gene
target_length = 1000

for chr in chr_list:
    os.makedirs(f"focused_sampling_length_control/{chr}", exist_ok=True) #folder for each chromosome
    
    #Getting the number of genes on this chromosome
    gtf_filtered = gtf[gtf['Chromosome'] == chr]
    number_genes = len(gtf_filtered)
    
    # Choosing a specific gene to focus on, for example gene number 10
    focused_gene_number = 10
    location = focused_gene_number
    
    
    my_automata = pa.automata_builder(gtf_file_path, chr, gene_number = location) # creating the automata
    
    list_mutated_samples = [] # list to store the mutated samples ( to store the instances of the mutated_sample class)
    
    for i in range(N):
        sample = sampling.mutated_sample(id = i, chromosome = chr, location = location, sequence = [], automata = my_automata, status = True)
        
        # ------------------------------ With Length control ------------------------------------------
        sample.generate_mutated_sample_length_control(target_length) #status updated here
        
        acceptance_status, score = my_automata.accepts(sample.sequence)

        # create a text file for the generated sequence
        file_name = f"{location}_sample_{i}.fasta"
        file_path = f"focused_sampling_length_control/{chr}/{file_name}"
        # save the generated sequence in a fasta file
        with open(file_path, "w") as f:
                f.write(f">{location}_sample_{i}\n") # Standard FASTA header
                f.write(sample.sequence)
                
        # save the metadata AND the file path to our database list
        database_fields.append({
                        "sample_id": i,
                        "chromosome": chr,
                        "gene_location": location, 
                        "sequence_length": len(sample.sequence),
                        "sample_status": sample.status, # if using controlled length used the status can be false 
                        "acceptance_score": score,
                        "file_path": f"{chr}/{file_name}"
                })
        list_mutated_samples.append(sample) # we store the instance of the mutated_sample class in the list
        
    # export the clean, readable database to a CSV so we can use it after 
    df_length_control = pd.DataFrame(database_fields)
    ## named manually 
    df_length_control.to_csv("focused_sampling_chr22_g10.csv", index=False)
            
print(f"The dataset for the chromosome(s) {chr_list} gene number {location} is ready! \n with {N} samples and target length: {target_length}")


The dataset for the chromosome(s) ['chr22'] gene number 10 is ready! 
 with 5000 samples and target length: 1000


It is faster than the free generation approach.

In [6]:
print(df_length_control.head())
true_only = df_length_control[df_length_control["sample_status"]== True]#those that have status = true
counts = true_only['sequence_length'].value_counts()

# new list of mutated_samples instances  with only the valid samples ( status = True)
list_mutated_samples_valid = [sample for sample in list_mutated_samples if sample.status == True]

print("-----------------------------------------------------Value counts on the True Status samples -----------------------------------------------------")
print(counts)
number_valid_samples = counts[1000]
print("---------------------------------------------------------Sampling Length Control Results -------------------------------------------------------------------")
print(f"                                          We generated {number_valid_samples} 1k-length valid samples from the whole 5k aim                                                 ")

   sample_id chromosome  gene_location  sequence_length  sample_status  \
0          0      chr22             10             1000           True   
1          1      chr22             10               34          False   
2          2      chr22             10             1000           True   
3          3      chr22             10             1000           True   
4          4      chr22             10               34          False   

   acceptance_score                file_path  
0       -156.083830  chr22/10_sample_0.fasta  
1         -1.772589  chr22/10_sample_1.fasta  
2       -130.371717  chr22/10_sample_2.fasta  
3       -142.220886  chr22/10_sample_3.fasta  
4         -1.772589  chr22/10_sample_4.fasta  
-----------------------------------------------------Value counts on the True Status samples -----------------------------------------------------
sequence_length
1000    4514
Name: count, dtype: int64
---------------------------------------------------------Sampling Lengt

In [7]:
#Lets check the false status samples how they be acting like trump 
false_only = df_length_control[df_length_control["sample_status"]== False]#those that have status = true
counts = false_only['sequence_length'].value_counts()
print(false_only.head())
print(counts)

    sample_id chromosome  gene_location  sequence_length  sample_status  \
1           1      chr22             10               34          False   
4           4      chr22             10               34          False   
9           9      chr22             10              998          False   
14         14      chr22             10              639          False   
15         15      chr22             10               99          False   

    acceptance_score                 file_path  
1          -1.772589   chr22/10_sample_1.fasta  
4          -1.772589   chr22/10_sample_4.fasta  
9        -126.618299   chr22/10_sample_9.fasta  
14        -83.002143  chr22/10_sample_14.fasta  
15         -9.514991  chr22/10_sample_15.fasta  
sequence_length
34     321
768      3
170      2
926      2
907      2
      ... 
722      1
587      1
321      1
371      1
132      1
Name: count, Length: 152, dtype: int64


## From a DNA Sequence to an Entropy Vector 
### (Encoding - Data Preparation for Neural Networks Training)
Using the vectorization.py file (the entropy_vector class)

explain the vector fields here 

In [ ]:
print(f"We have {len(true_only)} valid samples to encode.")
# vectorization of the valid samples
# result : list of entropy vectors for each sample

print(f"for example the sequence of the first valid sample instance \n {list_mutated_samples_valid[0].sequence} ")

k = 10
window_length = 100
list_entropy_vectors = [] # list to store the entropy vectors for all the dataset 

test1 = vec.entropy_vector(list_mutated_samples_valid[0], [])
test1.vectorize( k, window_length) # method call
print(test1.entropies)

# a problem in the entropy calculation method.


# creating a list of entropy vectors  from the list of valid samples instances
for sample in list_mutated_samples_valid: # sample in an instance of the mutated_sample class
    # creating an instance of the entropy_vector class for each sample
    sample_vector = vec.entropy_vector(sample, [])
    sample_vector.vectorize( k, window_length) # method call
    list_entropy_vectors.append(sample_vector.entropies)
    

print(f" entropy vectors calculated for the valid samples, each vector has 10 entropy measures (one per window) ")
      
print(f" The first entropy vector is: {list_entropy_vectors[0]}")        



We have 4514 valid samples to encode.
for example the sequence of the first valid sample instance 
 AAGCACTGGTGTCTCTGCAATCTAAATGCAACATAAATTGTCTCCCCTTCCCAGGGATCACATGACAAGTGGCGTGGGAGAAAACTTTCTAGAAACTTTTGCCTAGAATAACTCAAAAGTCATTGCTTATTGCAGAGAGTTCATAAACCCTGCCTGAAATACAAAAAAACTTAGCCTATATTAAATGTTGAATAGATCAAGCAGTCATAAAATCAGTACCTATAGTTACTCCATGCATATAGTATATGTACACTGTTTGTTTCTATTGATATTTTAGTTCTTTGCATTTTGTAGTTTGATGCTATAATTCTAATTACATTACCTCTCCTTCCATGACCCCTTTGTTTTTGCATTAAAAATACAAAAATGAGAAGGGTGATAAGTAATTCAGGTCCTCGAGCAAAAAGCATTCCAAAGAAAAGTTACAGTCAGGTCAATTTGCTATCTTGTGCAGATAACATATATAAGCAAGAAATATCAGTATTAGAAACCTATTTCCCCTCTACACGTGGGGATTATGTGTTGGGGCTCCAATCCCACATTTAAAAAATTTATTAAACTAGATAGTTTATATTGTTAATACTGCCAACATTGTGTCAAGAACTACCTGAGACTGAGAATGGGTATTTTATAAAGTGCTGTATCTGACAAAAAGTGCAGCTTTTTCATTGTTGTGCAGTATATATACATTTTCAACTTTAGGAATAGCCAAAAATACAAAAAAATTTACTATAGAAAAAGACTTTAATTGAAGTGTGGGAATCCAGAGACCCAGACTGTCTATGGTAAGGAAATACTGAAGATCTGGTAGCTTTCCCTCAGAATTATCCAGACTTTGTATGGCAAAAACCTGTCTCTAGTAAATATTTTATTGCTTTCCCACATCATCAGTTCTTACTA

In [ ]:
print(f"checking sequenceof the sample instance \n {list_mutated_samples_valid[0].sequence} ")